# 教程: 高级可视化 - 创建动画

## 目的
静态的图表可以很好地展示模拟的最终结果，但动画能够更直观、更生动地展示模型状态随时间动态演变的过程。本教程将展示如何为一维水力模型创建一个水面线变化的动画。

此笔记本将：
1.  建立一个带有闸门的水力模型，该模型具有有趣的动态变化过程。
2.  在循环中运行模拟，并在每一步将当前的水面线图保存为一张独立的图片。
3.  使用`imageio`库将所有保存的图片合成为一个GIF动画。
4.  展示如何在Jupyter Notebook中显示生成的动画。

## 1. 环境设置

我们导入所需的库。除了`matplotlib`，我们还需要`imageio`来创建GIF，以及`IPython.display`来在notebook中显示GIF。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
import os
import imageio
from IPython.display import Image

# 将项目根目录添加到Python路径
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname('__file__'), '..')))

from preissmann_model.model import HydraulicModel
from preissmann_model.reach import RiverReach
from preissmann_model.cross_section import RectangularCrossSection
from preissmann_model.structures import Gate

## 2. 模型和动画参数设置

我们建立一个与“闸门模型示例”中类似的河道。为了让动画效果更明显，我们模拟一个流量逐渐增大的过程，并动态调整闸门的开度。

In [ ]:
# --- 模型设置 ---
num_nodes = 21
reach_geom = RiverReach(
    cross_sections=[RectangularCrossSection(width=10) for _ in range(num_nodes)],
    lengths=np.full(num_nodes - 1, 250.0),
    slope=0.001,
    manning_n=0.03
)
gate = Gate(name="control_gate", node_index=10, opening_height=2.0, width=10)
river = HydraulicModel(name="animated_river", reach=reach_geom, dt=10.0, downstream_level=2.0, structures=[gate])
river.Q[:] = 5.0
for i in range(num_nodes):
    river.Z[i] = river.Z_bed[i] + 1.5

# --- 动画设置 ---
num_steps = 100
animation_dir = '../temp_animation_frames/'
output_gif_path = '../examples/results/wse_animation.gif'
if not os.path.exists(animation_dir):
    os.makedirs(animation_dir)
filenames = []

# --- 动态边界条件 ---
inflow_q = np.linspace(5.0, 50.0, num_steps)
gate_opening = np.concatenate([np.full(40, 2.0), np.linspace(2.0, 0.5, 30), np.full(30, 0.5)])

## 3. 生成动画帧

我们开始运行模拟。在每个时间步，我们：
1.  更新模型的边界条件（入流和闸门开度）。
2.  调用`step()`方法推进模拟。
3.  创建一个绘图，画出当前的水面线。
4.  将该图保存为一个PNG文件，并关闭绘图对象以释放内存。
5.  将文件名添加到一个列表中。

In [ ]:
print("开始生成动画帧...")
for i in range(num_steps):
    # 更新边界
    gate.opening_height = gate_opening[i]
    inflows = {'Q_inflow': inflow_q[i]}
    
    # 运行一步
    river.step(inflows, river.dt)
    
    # --- 创建绘图 ---
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(river.Z, 'b-o', label='Water Surface Elevation')
    ax.plot(river.Z_bed, 'k-', label='Bed Elevation')
    ax.axvline(gate.node_index, color='r', linestyle='--', label=f'Gate (Opening={gate.opening_height:.2f}m)')
    ax.set_ylim(0, 8) # 固定Y轴范围
    ax.set_title(f'Water Profile at Step {i+1}/{num_steps} (Inflow Q = {inflow_q[i]:.1f} m³/s)')
    ax.set_xlabel('Node Index')
    ax.set_ylabel('Elevation (m)')
    ax.legend(loc='upper left')
    ax.grid(True)
    
    # --- 保存并关闭 ---
    filename = f"{animation_dir}frame_{i:03d}.png"
    filenames.append(filename)
    plt.savefig(filename)
    plt.close(fig) # 必须关闭，否则会消耗大量内存
    
    if (i+1) % 10 == 0:
        print(f"  ...已生成 {i+1} 帧")

print("所有帧生成完毕。")

## 4. 合成并显示GIF

现在我们使用`imageio`来读取所有保存的PNG图片，并将它们合成为一个GIF文件。`duration`参数控制了每帧的显示时间。

In [ ]:
print(f"开始合成GIF到 {output_gif_path}...")
with imageio.get_writer(output_gif_path, mode='I', duration=0.1) as writer:
    for filename in filenames:
        image = imageio.imread(filename)
        writer.append_data(image)
print("GIF合成完毕。")

# (可选) 清理临时图片文件
for filename in filenames:
    os.remove(filename)
os.rmdir(animation_dir)
print("已删除临时图片文件。")

# 在Notebook中显示GIF
Image(url=output_gif_path)

动画生动地展示了随着上游入流的增加和闸门的关闭，闸门上游的水位是如何被逐渐壅高的。这种可视化方式对于理解模型的动态行为和调试复杂场景非常有用。